# ZeRO from scratch on 32 virtual GPUs

This notebook runs (or replays) every experiment behind `guide.md` and `README.md`, in order. Each "virtual GPU" is one CPU process started by `torchrun`; they talk over the gloo backend.

With `RERUN = False` (the default) each cell prints the logs that `runs.sh` already wrote under `runs/`, so the notebook opens in seconds. Set `RERUN = True` to run everything again: about 100 minutes on a fanless M4, which slows as it heats up.

In [ ]:
import json, os, subprocess
from IPython.display import Image, display

RERUN = False

def run(cmd, log=None):
    """Run a shell command and show its output; with RERUN off, show the saved log instead."""
    if RERUN or (log and not os.path.exists(log)):
        out = subprocess.run(cmd, shell=True, capture_output=True, text=True)
        text = out.stdout + out.stderr
        if log:
            os.makedirs(os.path.dirname(log), exist_ok=True)
            open(log, "w").write(text)
    else:
        text = open(log).read()
    print("\n".join(l for l in text.splitlines() if "VIRTUAL_ENV" not in l and not l.startswith(("W1", "*****"))))

## 1. The equivalence test

Each stage trains 5 steps in fp32 on 4 ranks, then on 3 ranks (3 does not divide the unit sizes, so padding gets exercised). Every stage must end with DDP's weights, and DDP with the weights of one process running `torch.optim.AdamW` on the whole global batch. Guide Part 12.

In [ ]:
run("uv run pytest test_zero.py -q -s", log="runs/test.log")

## 2. The scaling sweep

Every stage at N = 1, 2, 4, 8, 16, 32, five steps each. Memory and bytes sent stop changing after step 2. Guide Parts 14 and 16.

In [ ]:
run("uv run sweep.py --out runs/sweep --steps 5", log="runs/sweep.log")

## 3. The four main runs at N = 32

60 steps each, held-out loss every 30 steps. The first block of each output is the predicted memory bill, and the last block is what the ledger measured. Guide Part 13.

In [ ]:
for s in range(4):
    print(f"=============== stage {s} ===============")
    run(f"uv run torchrun --nproc_per_node 32 --master_port 2990{s} train.py --stage {s} --out runs/s{s}_n32 "
        f"--max-steps 60 --eval-every 30", log=f"runs/s{s}_n32.log")

## 4. One table: memory, bytes sent and time at N = 32

In [ ]:
rows = json.load(open("runs/sweep/sweep.json"))
print(f"{'stage':<8}{'params MB':>10}{'grads MB':>10}{'optim MB':>10}{'total MB':>10}{'peak MB':>10}{'sent MB':>10}{'sent/Psi':>10}{'ms/step':>10}")
for r in (r for r in rows if r["world"] == 32):
    st, sent = r["steady"], sum(r["sent_per_step"].values())
    print(f"{r['stage_name']:<8}{st['params']/1e6:>10.2f}{st['grads']/1e6:>10.2f}{st['optimizer']/1e6:>10.2f}"
          f"{sum(st.values())/1e6:>10.2f}{r['peak_total']/1e6:>10.2f}{sent/1e6:>10.2f}"
          f"{sent/(2*r['padded_params']):>10.3f}{1000*r['step_time']:>10.0f}")

## 5. Plots

In [ ]:
run("uv run plot.py --runs runs --out plots", log=None if RERUN else "runs/plot.log")
for name in ["memory", "scaling", "timeline", "comm", "work", "loss"]:
    print(name)
    display(Image(f"plots/{name}.png"))